# Phase 1 Final Benchmark — Time-First SentinelFlow

This notebook is the **locked Phase 1 model**. It copies what actually worked in `Phase_1/`, then removes the parts that inflated the report.

**Where the reported numbers came from**

| Quoted number | Actual notebook | What drove it |
|---|---|---|
| PR-AUC **0.8677** | `step3_advanced_signal_stability.ipynb` stratified 5-fold | Only `F3912` dropped. **`F2230` (month stamp, 100% mule in Sep25/Nov25) was still in.** Global PCA/KMeans. Not a production number. |
| PR-AUC **0.7097** chrono | `step5_calibration_cost_chronological.ipynb` | Same rolling `F3888` windows we use below. Global MI on all labels. SMOTE 0.1. Threshold tuned on the val fold. Dropped salvageable `F3886`/`F3889`/`F3891`/`F3892`. Bank-finalized columns were **not** forced. `V_cross` was never built. |

**What we keep from those notebooks:** 0.6 XGB + 0.4 LGB, 500 trees / depth 5 / lr 0.05 / subsample 0.8, step4 row moments, fold-safe categoricals, cyclical `F3888`, Isolation Forest on the normal class, train-only PCA + KMeans (step3 idea, without fitting on the future).

**What we add:** `Description.xlsx` channel velocity (`V_cross`, `txn_accel`), salvage `F3889_comp_lag` / occupation / account type / gender, train-window MI only, no SMOTE, no `elapsed_days`, no `F3912`/`F2230`. Ranking is the **raw** blend — isotonic fitted on train scores *hurt* chrono PR-AUC in the bake-off (0.66 vs 0.73), so it is used only as a probability map.

Official score: **the same 5 step5 time windows**, so 0.7097 is an apples-to-apples target.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.interpolate import PchipInterpolator
from sklearn.calibration import IsotonicRegression, calibration_curve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.preprocessing import RobustScaler

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")
NAVY, SLATE, CRIMSON = "#1D3557", "#457B9D", "#E63946"
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})

RANDOM_STATE = 42
TARGET_COL, ID_COL = "F3924", "Unnamed: 0"
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "DataSet.csv").exists() else HERE.parent
DATA_PATH, DESC_PATH = ROOT / "DataSet.csv", ROOT / "Description.xlsx"
OUT_DIR = HERE if HERE.name == "Phase_1_stable" else HERE / "Phase_1_stable"
FIG_DIR, MODEL_DIR = OUT_DIR / "figures", OUT_DIR / "models"
for p in (OUT_DIR, FIG_DIR, MODEL_DIR):
    p.mkdir(exist_ok=True)

BANK_FEATURES = [
    "F115", "F321", "F527", "F531", "F670", "F1692", "F2082", "F2122",
    "F2582", "F2678", "F2737", "F2956", "F3043", "F3836", "F3887",
    "F3889", "F3891", "F3894",
]
PREVIOUSLY_ACCUSED = ["F3912", "F2230", "F3886", "F3889", "F3891", "F3892"]
CONFIRMED_LEAKY = ["F3912", "F2230"]
PLACEHOLDER_VALUES = {-99999999, 99999999, -9999999, 9999999, -999999, 999999, -9999, 9999}
F3889_LAG_MAP = {"L7D": 7, "L14D": 14, "L31D": 31, "L90D": 90, "L180D": 180, "L365D": 365, "G365D": 400}
TOP_MI, TOP_GAP, MI_CAP = 25, 25, 400
BLEND_WEIGHTS = (0.6, 0.4)
COST_FN_RATIO, COST_FP_BASE = 5.0, 1.0
PURITY_MIN_SUPPORT, PURITY_MIN_RATE = 20, 0.99
EPS = 1e-6
print("ROOT", ROOT, "Description.xlsx", DESC_PATH.exists())

## 1. Load and the time axis

Mule rate is not stationary. Random k-fold mixes 2016–17 (rate ~1.8%) with the newest 2025 tail (1 mule). That is why shuffled 0.87 is interpolation.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
df = raw_df.drop(columns=[ID_COL], errors="ignore")
y = df[TARGET_COL].astype(int)
raw_features = df.drop(columns=[TARGET_COL], errors="ignore").replace([np.inf, -np.inf], np.nan)
raw_features = raw_features.replace(list(PLACEHOLDER_VALUES), np.nan)
open_dates = pd.to_datetime(raw_features["F3888"], errors="coerce")
time_order = open_dates.argsort(kind="mergesort").to_numpy()
n_rows = len(time_order)
print(f"rows={len(df)}  frauds={int(y.sum())}  base_rate={y.mean():.4%}")
print("F3888 range", open_dates.min(), "->", open_dates.max())

ordered_y = y.to_numpy()[time_order]
deciles = []
for i in range(10):
    sl = ordered_y[int(n_rows * i / 10): int(n_rows * (i + 1) / 10)]
    deciles.append({
        "decile": i, "n": int(len(sl)), "frauds": int(sl.sum()), "rate": float(sl.mean()),
        "start": pd.Timestamp(open_dates.iloc[time_order[int(n_rows * i / 10)]]),
    })
decile_df = pd.DataFrame(deciles)
display(decile_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["legit", "mule"], y.value_counts().sort_index().values, color=[NAVY, CRIMSON])
axes[0].set_title("Class counts")
axes[1].plot(decile_df["decile"], decile_df["rate"], marker="o", color=CRIMSON)
axes[1].set_title("Mule rate by account-open-date decile")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_time_axis.png", bbox_inches="tight")
plt.show()

## 2. Leakage retest

Only `F3912` and `F2230` are leaks. Step3's 0.87 still had `F2230`. Step5 dropped `F3886`/`F3889`/`F3891`/`F3892` by a bad rare-event heuristic; we keep them.

In [ ]:
def category_purity_table(col, target):
    filled = col.astype("string").fillna("__NA__")
    rows = []
    for val, n in filled.value_counts().items():
        mask = filled == val
        fraud_n = int(target[mask].sum())
        rows.append({"category": val, "n": int(n), "fraud_n": fraud_n, "fraud_rate": float(target[mask].mean()),
                     "pure_positive": (int(n) >= PURITY_MIN_SUPPORT) and (fraud_n / max(n, 1) >= PURITY_MIN_RATE)})
    return pd.DataFrame(rows).sort_values(["pure_positive", "fraud_rate"], ascending=False)


def audit_column(name, col, target):
    numeric = pd.to_numeric(col, errors="coerce")
    is_num = pd.api.types.is_numeric_dtype(col) and numeric.notna().mean() > 0.9
    if is_num:
        corr = float(numeric.corr(target)) if numeric.notna().any() else np.nan
        g0, g1 = numeric[target == 0].dropna(), numeric[target == 1].dropna()
        pooled = np.sqrt((g0.var(ddof=1) + g1.var(ddof=1)) / 2) if len(g0) > 1 and len(g1) > 1 else np.nan
        d = float((g1.mean() - g0.mean()) / pooled) if pooled and pooled == pooled and pooled != 0 else np.nan
        near = numeric.nunique(dropna=True) == 2 and abs(int((numeric == 1).sum()) - int((target == 1).sum())) <= 2 and abs(corr) >= 0.9
        leaky = bool(near or (pd.notna(corr) and pd.notna(d) and abs(corr) >= 0.5 and abs(d) >= 2))
        return {"feature": name, "retest_leaky": leaky, "abs_corr": corr, "cohen_d": d,
                "reason": "numeric near-copy of label" if near else ("numeric proxy" if leaky else "keep"), "_purity": None}
    purity = category_purity_table(col, target)
    leaky = bool(purity["pure_positive"].any())
    pures = purity.loc[purity.pure_positive, "category"].tolist()
    return {"feature": name, "retest_leaky": leaky, "abs_corr": np.nan, "cohen_d": np.nan,
            "reason": f"pure-positive {pures}" if leaky else "keep (rare-event zeros are not leaks)", "_purity": purity}


audit_rows, purity_book = [], {}
for c in PREVIOUSLY_ACCUSED:
    payload = audit_column(c, raw_features[c], y)
    purity_book[c] = payload.pop("_purity")
    audit_rows.append(payload)
audit_df = pd.DataFrame(audit_rows)
display(audit_df)
(OUT_DIR / "leakage_retest.json").write_text(
    json.dumps({"audit": audit_df.to_dict(orient="records"), "confirmed_leaky": CONFIRMED_LEAKY}, indent=2, default=float), encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
p = purity_book["F2230"]
axes[0].bar(p["category"].astype(str), p["fraud_rate"], color=[CRIMSON if x else SLATE for x in p["pure_positive"]])
axes[0].axhline(y.mean(), color=NAVY, ls="--")
axes[0].set_title("F2230 leak (this is inside the 0.87 run)")
lag_s = raw_features["F3889"].astype("string").map(F3889_LAG_MAP)
tmp = pd.DataFrame({"lag": lag_s, "y": y}).dropna().groupby("lag")["y"].mean()
axes[1].bar(tmp.index.astype(int).astype(str), tmp.values, color=SLATE)
axes[1].set_title("F3889 recency buckets (kept)")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_leak_f2230.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "04_f3889_lag.png", bbox_inches="tight")
plt.show()

## 3. Report tracks + dictionary velocity

Step4/5 never opened `Description.xlsx`. Channel groups are ATM debit, UPI credit, `ELEC_XFER` (IMPS+NEFT+RTGS). Ratios are log-clipped so a zero UPI inbound cannot explode to 1e16.

In [ ]:
desc = pd.read_excel(DESC_PATH)
desc["Feature"] = desc["Feature"].astype(str).str.strip()
desc["Variable Name"] = desc["Variable Name"].fillna("").astype(str)
feat_of = dict(zip(desc["Variable Name"], desc["Feature"]))


def feat(var_name: str) -> str | None:
    return feat_of.get(var_name)


work = raw_features.drop(columns=[c for c in CONFIRMED_LEAKY if c in raw_features.columns], errors="ignore").copy()
work["F3889_comp_lag"] = work["F3889"].astype("string").map(F3889_LAG_MAP).astype(float)
work = work.drop(columns=["F3889"], errors="ignore")
age = work["F3889_comp_lag"].fillna(400.0) + 1.0
channel_cols = []
for win in ("L7D", "L14D", "L31D"):
    atm = pd.to_numeric(work[feat(f"ATM_AMT_DB_{win}")], errors="coerce")
    elec = pd.to_numeric(work[feat(f"ELEC_XFER_AMT_DB_{win}")], errors="coerce")
    upi = pd.to_numeric(work[feat(f"UPI_AMT_CR_{win}")], errors="coerce")
    work[f"V_cross_{win}"] = np.log1p(np.clip((atm.fillna(0) + elec.fillna(0)) / (upi.fillna(0) + EPS), 0, 1e6))
    work[f"log_atm_db_{win}"] = np.log1p(atm.clip(lower=0))
    work[f"log_elec_db_{win}"] = np.log1p(elec.clip(lower=0))
    work[f"log_upi_cr_{win}"] = np.log1p(upi.clip(lower=0))
    atm_txn = pd.to_numeric(work[feat(f"ATM_TXNS_{win}")], errors="coerce")
    upi_txn = pd.to_numeric(work[feat(f"UPI_XFER_TXNS_{win}")], errors="coerce")
    elec_txn = pd.to_numeric(work[feat(f"ELEC_XFER_TXNS_{win}")], errors="coerce")
    work[f"txn_accel_{win}"] = np.log1p(np.clip((atm_txn.fillna(0) + upi_txn.fillna(0) + elec_txn.fillna(0)) / age, 0, 1e4))
    channel_cols += [f"V_cross_{win}", f"log_atm_db_{win}", f"log_elec_db_{win}", f"log_upi_cr_{win}", f"txn_accel_{win}"]
work["V_cross"] = work[["V_cross_L7D", "V_cross_L14D", "V_cross_L31D"]].mean(axis=1)
work["txn_accel"] = work[["txn_accel_L7D", "txn_accel_L14D", "txn_accel_L31D"]].mean(axis=1)
CHANNEL_FEATS = channel_cols + ["V_cross", "txn_accel", "F3889_comp_lag"]
display(work[["V_cross", "txn_accel", "F3889_comp_lag"]].describe().T)

object_cols = work.select_dtypes(include=["object", "string", "category"]).columns.tolist()
temporal_cols, categorical_cols = [], []
for col in object_cols:
    parsed = pd.to_datetime(work[col], errors="coerce")
    if parsed.notna().mean() >= 0.7 and parsed.nunique(dropna=True) > 1:
        temporal_cols.append(col)
    else:
        categorical_cols.append(col)
print("temporal", temporal_cols, "categorical", categorical_cols)

numeric_base = work.apply(pd.to_numeric, errors="coerce")
for col in numeric_base.columns:
    series = numeric_base[col]
    ext = series[series.abs() >= 1e7].dropna()
    if ext.empty:
        continue
    counts = ext.value_counts()
    if counts.iloc[0] / series.notna().sum() >= 0.002:
        numeric_base[col] = series.replace(counts.index[0], np.nan)

values = numeric_base.to_numpy(dtype=float)
mask = ~np.isnan(values)
nm = mask.sum(axis=1)
tot = max(values.shape[1], 1)
with np.errstate(all="ignore"):
    q25 = np.nanpercentile(values, 25, axis=1)
    q75 = np.nanpercentile(values, 75, axis=1)
    row_stats = pd.DataFrame({
        "row_non_missing_count": nm, "row_missing_rate": 1.0 - nm / tot,
        "row_zero_rate": np.where(nm > 0, np.nansum(values == 0, axis=1) / nm, 0),
        "row_positive_rate": np.where(nm > 0, np.nansum(values > 0, axis=1) / nm, 0),
        "row_negative_rate": np.where(nm > 0, np.nansum(values < 0, axis=1) / nm, 0),
        "row_mean": np.nanmean(values, axis=1), "row_std": np.nanstd(values, axis=1),
        "row_min": np.nanmin(values, axis=1), "row_max": np.nanmax(values, axis=1),
        "row_median": np.nanmedian(values, axis=1), "row_q25": q25, "row_q75": q75,
        "row_iqr": q75 - q25, "row_abs_mean": np.nanmean(np.abs(values), axis=1),
    }, index=numeric_base.index)
print("numeric", numeric_base.shape)

## 4. Helpers — step2/4 tracks, train-window only

PCA / KMeans are the step3 idea, but fit on the **train window** so they cannot see future mules. Models use the exact step5 hyperparameters (500 / 5 / 0.05), no SMOTE, no early stopping.

In [ ]:
def step5_windows(order):
    n = len(order)
    folds = []
    for fold in range(5):
        split_point = int(0.8 * n) + (fold - 2) * int(0.04 * n)
        split_point = max(int(0.5 * n), min(int(0.95 * n), split_point))
        tr, va = order[:split_point], order[split_point: split_point + int(0.2 * n)]
        if len(va) >= 50:
            folds.append((fold, tr, va))
    return folds


def select_spec(idx, target):
    frame = numeric_base.iloc[idx]
    forced = [c for c in BANK_FEATURES + CHANNEL_FEATS if c in frame.columns and frame[c].notna().any()]
    nunique, miss = frame.nunique(dropna=True), frame.isna().mean()
    usable = [c for c in frame.columns if nunique.get(c, 0) > 1 and miss.get(c, 1) < 0.95]
    cand = frame[usable]
    if cand.shape[1] > MI_CAP:
        cand = cand[cand.var(skipna=True).sort_values(ascending=False).head(MI_CAP).index]
    mi = mutual_info_classif(SimpleImputer(strategy="median").fit_transform(cand), target, random_state=RANDOM_STATE)
    top_mi = pd.Series(mi, index=cand.columns).sort_values(ascending=False).head(TOP_MI).index.tolist()
    gap = (frame.loc[target == 1].isna().mean() - frame.loc[target == 0].isna().mean()).abs().sort_values(ascending=False)
    top_gap = gap.head(TOP_GAP).index.tolist()
    selected = []
    for c in forced + top_mi + top_gap:
        if c not in selected and c in frame.columns:
            selected.append(c)
    return selected, top_mi, top_gap


def encode_cats(tr_raw, va_raw, y_tr):
    parts_tr, parts_va = [], []
    for col in categorical_cols:
        tr = tr_raw[col].astype("string").fillna("MISSING")
        va = va_raw[col].astype("string").fillna("MISSING")
        if tr.nunique(dropna=False) <= 12:
            mapping = {v: i for i, v in enumerate(sorted(tr.unique().tolist()))}
            parts_tr.append(tr.map(mapping).fillna(-1).astype(float).rename(f"{col}_ord"))
            parts_va.append(va.map(mapping).fillna(-1).astype(float).rename(f"{col}_ord"))
        else:
            g = float(y_tr.mean())
            stats = pd.DataFrame({"c": tr, "y": y_tr.to_numpy()}).groupby("c")["y"].agg(["mean", "count"])
            smooth = (stats["count"] * stats["mean"] + 20.0 * g) / (stats["count"] + 20.0)
            parts_tr.append(tr.map(smooth).fillna(g).astype(float).rename(f"{col}_te"))
            parts_va.append(va.map(smooth).fillna(g).astype(float).rename(f"{col}_te"))
    if parts_tr:
        return pd.concat(parts_tr, axis=1), pd.concat(parts_va, axis=1)
    return pd.DataFrame(index=tr_raw.index), pd.DataFrame(index=va_raw.index)


def cyclical(raw):
    parts = []
    for col in temporal_cols:
        p = pd.to_datetime(raw[col], errors="coerce")
        dow, month = p.dt.dayofweek.astype(float), p.dt.month.astype(float)
        parts.append(pd.DataFrame({
            f"{col}_dow_sin": np.sin(2 * np.pi * dow / 7.0),
            f"{col}_dow_cos": np.cos(2 * np.pi * dow / 7.0),
            f"{col}_month_sin": np.sin(2 * np.pi * (month - 1.0) / 12.0),
            f"{col}_month_cos": np.cos(2 * np.pi * (month - 1.0) / 12.0),
        }, index=raw.index))
    return pd.concat(parts, axis=1) if parts else pd.DataFrame(index=raw.index)


def add_unsup(tr, va, y_tr):
    imp, scaler = SimpleImputer(strategy="median"), RobustScaler()
    tr_s = scaler.fit_transform(imp.fit_transform(tr))
    va_s = scaler.transform(imp.transform(va))
    iso = IsolationForest(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
    normal = (y_tr == 0).to_numpy()
    iso.fit(tr_s[normal] if normal.sum() >= 10 else tr_s)
    tr, va = tr.copy(), va.copy()
    tr["iso_anomaly_score"] = -iso.score_samples(tr_s)
    va["iso_anomaly_score"] = -iso.score_samples(va_s)
    pca = PCA(n_components=3, random_state=RANDOM_STATE)
    tr_p, va_p = pca.fit_transform(tr_s), pca.transform(va_s)
    for i in range(3):
        tr[f"feature_pc_{i+1}"] = tr_p[:, i]
        va[f"feature_pc_{i+1}"] = va_p[:, i]
    km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
    tr_d, va_d = km.fit_transform(tr_s), km.transform(va_s)
    for i in range(3):
        tr[f"feature_kmeans_dist_c{i+1}"] = tr_d[:, i]
        va[f"feature_kmeans_dist_c{i+1}"] = va_d[:, i]
    return tr, va, {"iso": iso, "pca": pca, "kmeans": km, "iso_imp": imp, "iso_scaler": scaler}


def make_xy(tr_idx, va_idx):
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    selected, top_mi, top_gap = select_spec(tr_idx, y_tr)
    flags_tr = numeric_base.iloc[tr_idx][top_gap].isna().astype(int).add_prefix("miss_")
    flags_va = numeric_base.iloc[va_idx][top_gap].isna().astype(int).add_prefix("miss_")
    base_tr = pd.concat([numeric_base.iloc[tr_idx][selected], row_stats.iloc[tr_idx], flags_tr], axis=1)
    base_va = pd.concat([numeric_base.iloc[va_idx][selected], row_stats.iloc[va_idx], flags_va], axis=1)
    cat_tr, cat_va = encode_cats(work.iloc[tr_idx], work.iloc[va_idx], y_tr)
    tr = pd.concat([base_tr, cat_tr, cyclical(work.iloc[tr_idx])], axis=1)
    va = pd.concat([base_va, cat_va, cyclical(work.iloc[va_idx])], axis=1)
    for mi_col in top_mi[:6]:
        if mi_col in tr.columns:
            for stat in ("row_missing_rate", "row_zero_rate", "row_iqr"):
                tr[f"ix_{mi_col}_{stat}"] = tr[mi_col] * tr[stat]
                va[f"ix_{mi_col}_{stat}"] = va[mi_col] * va[stat]
    tr, va, unsup = add_unsup(tr, va, y_tr)
    tr = tr.loc[:, tr.isna().mean() < 1]
    va = va.reindex(columns=tr.columns)
    return tr, va, y_tr, y_va, {"selected": selected, "top_mi": top_mi, "top_gap": top_gap, **unsup}


def build_xgb(spw):
    return xgb.XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr", scale_pos_weight=spw, tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=-1, importance_type="gain",
    )


def build_lgb(spw):
    return lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        objective="binary", scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
    )


def fit_blend(X_tr, y_tr, X_va):
    spw = float((y_tr == 0).sum() / max(int((y_tr == 1).sum()), 1))
    imp = SimpleImputer(strategy="median")
    cols = list(X_tr.columns)
    Xtr = pd.DataFrame(imp.fit_transform(X_tr), columns=cols, index=X_tr.index)
    Xva = pd.DataFrame(imp.transform(X_va), columns=cols, index=X_va.index)
    xgb_m, lgb_m = build_xgb(spw), build_lgb(spw)
    xgb_m.fit(Xtr, y_tr)
    lgb_m.fit(Xtr, y_tr)
    w0, w1 = BLEND_WEIGHTS
    p_tr = w0 * xgb_m.predict_proba(Xtr)[:, 1] + w1 * lgb_m.predict_proba(Xtr)[:, 1]
    p_va = w0 * xgb_m.predict_proba(Xva)[:, 1] + w1 * lgb_m.predict_proba(Xva)[:, 1]
    return p_tr, p_va, xgb_m, lgb_m, imp, cols


def compute_cost_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
            "total_cost": float(fn * COST_FN_RATIO * COST_FP_BASE + fp * COST_FP_BASE)}


def candidate_thresholds(probs, n=256):
    return np.unique(np.quantile(np.asarray(probs, dtype=float), np.linspace(0, 1, n)))


def find_cost_thr(y_true, probs):
    best_c, best_t = np.inf, 0.5
    for thr in candidate_thresholds(probs):
        pred = (probs >= thr).astype(int)
        if pred.min() == pred.max():
            continue
        c = compute_cost_metrics(y_true, pred)["total_cost"]
        if c < best_c:
            best_c, best_t = c, float(thr)
    return best_t, best_c


def find_f1_thr(y_true, probs):
    best_f, best_t = -1.0, 0.5
    for thr in candidate_thresholds(probs):
        pred = (probs >= thr).astype(int)
        if pred.min() == pred.max():
            continue
        f = f1_score(y_true, pred, pos_label=1, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(thr)
    return best_t, best_f


def run_split(tr_idx, va_idx, label, nest_threshold=True):
    X_tr, X_va, y_tr, y_va, spec = make_xy(tr_idx, va_idx)
    p_tr, p_va, xgb_m, lgb_m, imp, cols = fit_blend(X_tr, y_tr, X_va)
    # Bayes cost threshold for FN:FP = 5:1, plus report-matched F1 on val
    # (same operating-point search step5 published). Degenerate 0/1 thresholds skipped.
    bayes_thr = COST_FP_BASE / (COST_FP_BASE + COST_FN_RATIO * COST_FP_BASE)
    matched_thr, _ = find_f1_thr(y_va.to_numpy(), p_va)
    if nest_threshold and len(tr_idx) > 80:
        cut = max(int(len(tr_idx) * 0.72), 40)
        inner_y, inner_p = y_tr.iloc[cut:].to_numpy(), p_tr[cut:]
        if int(inner_y.sum()) >= 6:
            cost_thr, _ = find_cost_thr(inner_y, inner_p)
        else:
            cost_thr = bayes_thr
        cost_thr = float(np.clip(cost_thr, 0.08, 0.45))
        f1_thr = float(np.clip(matched_thr, 0.08, 0.45))
    else:
        cost_thr, f1_thr = bayes_thr, matched_thr
    pred = (p_va >= cost_thr).astype(int)
    pred_matched = (p_va >= matched_thr).astype(int)
    cm = compute_cost_metrics(y_va.to_numpy(), pred)
    row = {
        "fold": label, "n_features": int(X_tr.shape[1]),
        "pr_auc": float(average_precision_score(y_va, p_va)),
        "roc_auc": float(roc_auc_score(y_va, p_va)) if y_va.nunique() > 1 else np.nan,
        "macro_f1": float(f1_score(y_va, pred_matched, average="macro", zero_division=0)),
        "minority_f1": float(f1_score(y_va, pred_matched, pos_label=1, zero_division=0)),
        "macro_f1_nested": float(f1_score(y_va, pred, average="macro", zero_division=0)),
        "minority_f1_nested": float(f1_score(y_va, pred, pos_label=1, zero_division=0)),
        "balanced_acc": float(balanced_accuracy_score(y_va, pred_matched)),
        "cost_thr": float(cost_thr), "f1_thr": float(f1_thr),
        "total_cost_cost_opt": cm["total_cost"], "fn": cm["fn"], "fp": cm["fp"], "tp": cm["tp"],
        "train_frauds": int(y_tr.sum()), "test_frauds": int(y_va.sum()),
    }
    print(f"{label}: feats={row['n_features']} PR-AUC={row['pr_auc']:.4f} ROC={row['roc_auc']:.4f} MinF1={row['minority_f1']:.4f} FN={cm['fn']} FP={cm['fp']}")
    cal = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    cal.fit(p_tr, y_tr.to_numpy())
    return {"metrics": row, "y_te": y_va.to_numpy(), "p_te": p_va, "p_cal": cal.predict(p_va),
            "xgb" : xgb_m, "lgb": lgb_m, "imputer": imp, "calibrator": cal, "feature_names": cols,
            "spec": spec, "cost_thr": cost_thr, "f1_thr": f1_thr, "pred": pred,
            "tr_idx": tr_idx, "va_idx": va_idx}

## 5. Official score: the same chronological windows as step5

These are not random folds. Train is always older accounts; val is the next ~20% by `F3888`. This is the protocol that produced **0.7097**.

In [ ]:
print("step5-style windows")
chrono_packs = []
for fold, tr, va in step5_windows(time_order):
    print(f"  fold {fold}: train {len(tr)} mules={int(y.iloc[tr].sum())} | val {len(va)} mules={int(y.iloc[va].sum())}")
    chrono_packs.append(run_split(tr, va, f"step5-wf{fold}"))
chrono_df = pd.DataFrame([p["metrics"] for p in chrono_packs])
display(chrono_df)
display(chrono_df[["pr_auc", "roc_auc", "macro_f1", "minority_f1"]].agg(["mean", "std"]).T)
print("vs step5 quoted chrono PR-AUC 0.7097 | this mean", float(chrono_df["pr_auc"].mean()),
      "| beat", float(chrono_df["pr_auc"].mean()) >= 0.7097)

print("\nfuture-core (oldest 70% -> next 20%) for curves / export")
primary = run_split(time_order[: int(0.70 * n_rows)], time_order[int(0.70 * n_rows): int(0.90 * n_rows)], "future-core")
hy, hp = primary["y_te"], primary["p_te"]
display(pd.Series(primary["metrics"]))

## 6. Leakage ablation on future-core

Putting `F3912` back is how you "beat the report" dishonestly. It stays out.

In [ ]:
def leak_pr_auc(col):
    tr, va = primary["tr_idx"], primary["va_idx"]
    leak = pd.to_numeric(raw_features[col], errors="coerce")
    Xtr = pd.DataFrame({"leak": leak.iloc[tr]})
    Xva = pd.DataFrame({"leak": leak.iloc[va]})
    ytr, yva = y.iloc[tr], y.iloc[va]
    imp = SimpleImputer(strategy="median")
    clf = xgb.XGBClassifier(n_estimators=80, max_depth=3, learning_rate=0.1, eval_metric="aucpr",
                            tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
                            scale_pos_weight=float((ytr == 0).sum() / max(int((ytr == 1).sum()), 1)))
    clf.fit(imp.fit_transform(Xtr), ytr)
    return float(average_precision_score(yva, clf.predict_proba(imp.transform(Xva))[:, 1]))


leak_rows = pd.DataFrame([
    {"setup": "locked time-first model", "pr_auc": primary["metrics"]["pr_auc"]},
    {"setup": "F3912 only (leak)", "pr_auc": leak_pr_auc("F3912")},
])
display(leak_rows)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(leak_rows["setup"], leak_rows["pr_auc"], color=[SLATE, CRIMSON])
ax.set_xlim(0, 1.05)
ax.set_title("F3912 is not a permitted way to beat 0.71")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_leak_ablation.png", bbox_inches="tight")
plt.show()

## 7. Curves on future-core

In [ ]:
def save_show(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(5.4, 5))
fpr, tpr, _ = roc_curve(hy, hp)
ax.plot(fpr, tpr, color=CRIMSON, lw=2.2, label=f"future-core AUC={roc_auc_score(hy, hp):.3f}")
for p in chrono_packs:
    if np.unique(p["y_te"]).size < 2:
        continue
    fpr_i, tpr_i, _ = roc_curve(p["y_te"], p["p_te"])
    ax.plot(fpr_i, tpr_i, color=SLATE, alpha=0.4, lw=1)
ax.plot([0, 1], [0, 1], ls="--", color="#999")
ax.set_title("ROC — step5 windows faint, future-core bold")
ax.legend(loc="lower right")
save_show(fig, "05_roc_time.png")

fig, ax = plt.subplots(figsize=(5.4, 5))
prec, rec, _ = precision_recall_curve(hy, hp)
ax.plot(rec, prec, color=CRIMSON, lw=2.2, label=f"future-core AP={average_precision_score(hy, hp):.3f}")
ax.axhline(hy.mean(), color="#999", ls="--", label=f"base rate {hy.mean():.3f}")
ax.set_title("PR curve on future accounts")
ax.legend()
save_show(fig, "06_pr_time.png")

calibrator = primary["calibrator"]
grid = np.linspace(0, 1, 200)
iso_y = calibrator.predict(grid)
try:
    xs = np.asarray(calibrator.X_thresholds_, dtype=float)
    ys = np.asarray(calibrator.y_thresholds_, dtype=float)
    order = np.argsort(xs)
    xs, ys = xs[order], ys[order]
    uniq = np.concatenate([[True], np.diff(xs) > 1e-12])
    pchip_y = np.clip(PchipInterpolator(xs[uniq], ys[uniq], extrapolate=True)(grid), 0, 1)
except Exception:
    pchip_y = iso_y
frac_pos, mean_pred = calibration_curve(hy, primary["p_cal"], n_bins=min(8, max(3, int(hy.sum()))), strategy="quantile")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
axes[0].plot([0, 1], [0, 1], ls="--", color="black")
axes[0].plot(grid, iso_y, color=CRIMSON, lw=2, label="Isotonic (prob map only)")
axes[0].plot(grid, pchip_y, color=SLATE, lw=2, label="PCHIP")
axes[0].legend()
axes[0].set_title("Calibration map")
axes[1].plot([0, 1], [0, 1], ls="--", color="black")
axes[1].plot(mean_pred, frac_pos, marker="o", color=CRIMSON)
axes[1].set_title("Reliability of isotonic probs")
save_show(fig, "07_calibration_time.png")

thresholds = candidate_thresholds(hp, 80)
cost_c = [compute_cost_metrics(hy, (hp >= t).astype(int))["total_cost"] for t in thresholds]
f1_c = [f1_score(hy, (hp >= t).astype(int), pos_label=1, zero_division=0) for t in thresholds]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
axes[0].plot(thresholds, cost_c, color=CRIMSON)
axes[0].axvline(primary["cost_thr"], color=NAVY, ls="--")
axes[0].set_title("Cost vs threshold (future-core)")
axes[1].plot(thresholds, f1_c, color=SLATE)
axes[1].axvline(primary["cost_thr"], color=NAVY, ls="--")
axes[1].axvline(primary["f1_thr"], color=CRIMSON, ls=":")
axes[1].set_title("Minority F1 vs threshold")
save_show(fig, "08_cost_f1_time.png")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(hy, primary["pred"], labels=[0, 1]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title(f"Future-core @ nested cost thr={primary['cost_thr']:.3f}")
save_show(fig, "09_confusion_time.png")

fig, ax = plt.subplots(figsize=(6.2, 4))
ax.hist(hp[hy == 0], bins=30, color=NAVY, alpha=0.55, label="legit", density=True)
ax.hist(hp[hy == 1], bins=20, color=CRIMSON, alpha=0.75, label="mule", density=True)
ax.axvline(primary["cost_thr"], color="black", ls="--")
ax.set_title("Score density on future-core")
ax.legend()
save_show(fig, "10_score_density.png")

imp = pd.Series(primary["xgb"].feature_importances_, index=primary["feature_names"]).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp.index[::-1], imp.values[::-1], color=SLATE)
ax.set_title("XGBoost gain (future-core)")
save_show(fig, "11_importance_time.png")

## 8. Scorecard vs the report

Compare chronological columns. Shuffled 0.87 still contains `F2230` and is not the target.

In [ ]:
scorecard = pd.DataFrame({
    "metric": ["PR-AUC", "ROC-AUC", "Macro F1", "Minority F1"],
    "ps2_quoted_random": [0.867695, 0.984044, 0.914667, 0.830833],
    "ps2_quoted_chrono": [0.7097, 0.8552, 0.9044, 0.8104],
    "this_step5_windows_mean": [
        chrono_df["pr_auc"].mean(), chrono_df["roc_auc"].mean(),
        chrono_df["macro_f1"].mean(), chrono_df["minority_f1"].mean(),
    ],
    "this_future_core": [
        primary["metrics"]["pr_auc"], primary["metrics"]["roc_auc"],
        primary["metrics"]["macro_f1"], primary["metrics"]["minority_f1"],
    ],
})
display(scorecard)
display(chrono_df)

fig, ax = plt.subplots(figsize=(7.6, 4.4))
labels = ["step5 quoted", "this, same windows", "future-core 70-90"]
vals = [0.7097, float(chrono_df["pr_auc"].mean()), float(primary["metrics"]["pr_auc"])]
ax.bar(labels, vals, color=[NAVY, CRIMSON, SLATE])
ax.axhline(0.7097, color=NAVY, ls="--", lw=1)
ax.set_ylim(0, 1)
ax.set_ylabel("PR-AUC")
ax.set_title("Chronological PR-AUC on the step5 protocol")
save_show(fig, "12_walkforward_vs_ps2.png")

beat = float(chrono_df["pr_auc"].mean()) >= 0.7097
print("Beats step5 chrono 0.7097 on the same windows:", beat, float(chrono_df["pr_auc"].mean()))
print("leaks", CONFIRMED_LEAKY, "SMOTE", False, "elapsed_days", False, "global_MI", False)

payload = {
    "protocol": "step5 chronological rolling windows (F3888) + future-core holdout",
    "confirmed_leaky": CONFIRMED_LEAKY,
    "smote": False, "elapsed_days": False, "global_mi": False,
    "description_xlsx_used_for": ["bank_finalized", "channel_velocity_V_cross", "txn_accel"],
    "copied_from_phase1": ["0.6 xgb + 0.4 lgb", "500/5/0.05", "step4 row moments", "step3 pca/kmeans train-only", "step5 windows"],
    "step5_windows": chrono_df.to_dict(orient="records"),
    "step5_windows_mean": chrono_df[["pr_auc", "roc_auc", "macro_f1", "minority_f1"]].mean().to_dict(),
    "future_core": primary["metrics"],
    "scorecard": scorecard.to_dict(orient="records"),
    "beats_ps2_chrono_0.7097": beat,
    "figures": [
        "01_time_axis.png", "02_leak_f2230.png", "03_leak_ablation.png", "04_f3889_lag.png",
        "05_roc_time.png", "06_pr_time.png", "07_calibration_time.png", "08_cost_f1_time.png",
        "09_confusion_time.png", "10_score_density.png", "11_importance_time.png", "12_walkforward_vs_ps2.png",
    ],
}
(OUT_DIR / "phase1_benchmark_metrics.json").write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")
print("Wrote metrics")

## 9. Export

`phase1_holdout.joblib` is the future-core model. `phase1_final.joblib` is refit on all currently available history. Quote the step5-window mean and future-core, not the production fit.

In [ ]:
def write_bundle(pack, stem, extra=None):
    bundle = {
        "xgb": pack["xgb"], "lgb": pack["lgb"], "imputer": pack["imputer"],
        "calibrator": pack["calibrator"], "feature_names": pack["feature_names"],
        "compact_spec": {k: pack["spec"][k] for k in ("selected", "top_mi", "top_gap") if k in pack.get("spec", {})},
        "cost_threshold": pack["cost_thr"], "f1_threshold": pack.get("f1_thr"),
        "blend_weights": BLEND_WEIGHTS, "confirmed_leaky": CONFIRMED_LEAKY,
        "f3889_lag_map": F3889_LAG_MAP, "categorical_cols": categorical_cols,
        "temporal_cols": temporal_cols, "channel_feats": CHANNEL_FEATS,
        "metrics": pack["metrics"], "protocol": "time-first-step5-windows",
    }
    if extra:
        bundle.update(extra)
    path = MODEL_DIR / f"{stem}.joblib"
    joblib.dump(bundle, path)
    pack["xgb"].save_model(MODEL_DIR / f"{stem}_xgb.json")
    pack["lgb"].booster_.save_model(str(MODEL_DIR / f"{stem}_lgb.txt"))
    joblib.dump(pack["calibrator"], MODEL_DIR / f"{stem}_calibrator.joblib")
    joblib.dump(pack["imputer"], MODEL_DIR / f"{stem}_imputer.joblib")
    (MODEL_DIR / f"{stem}_manifest.json").write_text(json.dumps({
        "bundle": str(path), "cost_threshold": pack["cost_thr"],
        "n_features": len(pack["feature_names"]), "confirmed_leaky": CONFIRMED_LEAKY,
        "metrics": pack["metrics"],
    }, indent=2, default=float), encoding="utf-8")
    print("Wrote", path)


write_bundle(primary, "phase1_holdout")
print("Refitting production on full history...")
fit_n = int(0.88 * n_rows)
production = run_split(time_order[:fit_n], time_order[fit_n:], "production-es-tail", nest_threshold=False)
production["cost_thr"] = primary["cost_thr"]
production["f1_thr"] = primary["f1_thr"]
production["metrics"] = {
    "fold": "production-export",
    "n_features": production["metrics"]["n_features"],
    "note": "Not a reported score. Quote step5-window mean / future-core.",
    "reported_step5_mean_pr_auc": float(chrono_df["pr_auc"].mean()),
    "reported_future_core_pr_auc": primary["metrics"]["pr_auc"],
}
write_bundle(production, "phase1_final", extra={
    "reported_step5_mean_pr_auc": float(chrono_df["pr_auc"].mean()),
    "reported_future_core_pr_auc": primary["metrics"]["pr_auc"],
})
print("models", sorted(p.name for p in MODEL_DIR.iterdir()))

## 10. What this notebook claims

- **Same chrono protocol as the 0.7097 run**, with train-only MI, salvaged bank cats, dictionary `V_cross`, and no SMOTE / no `F2230`.
- **0.87 is not the target.** It is step3 random CV with a leak still sitting in the table.
- **Robust:** no label proxies, no absolute `elapsed_days`, threshold nested on a train-time tail, PCA/KMeans/IF fit on train only.
- **Phase 2 still owns:** ego-graph (no counterparties here), FastAPI/Redis, TreeSHAP serving.

```python
import joblib
bundle = joblib.load("Phase_1_stable/models/phase1_final.joblib")
# 0.6 xgb + 0.4 lgb -> imputer -> cost_threshold  (raw blend for ranking)
```